# Project 2 — CIFAR-10 Classification Network

**Students**
- Dawod Ghifari — 520140154
- Hilal Chamtie — 540749283
- Darin Li — 500048292
- Akshar Hossain — 540823996

**Task summary (from `project_2_2026.pdf`)**
1. Download CIFAR-10.
2. Split into training and testing using an **80% : 20%** ratio.
3. Design a neural network for 10-class image classification.
4. Train and evaluate.

**Objectives**
- **(i)** Achieve **> 80% test accuracy** (primary objective).
- **(ii)** After the initial training, improve the architecture / training strategy
  so that the model reaches **≥ 70% test accuracy within 1 minute of training on a T4 GPU**.

**Task 2 — Question 1** is answered at the bottom of this notebook.

> **Runtime:** Google Colab with a **T4 GPU**. Before running, go to
> `Runtime → Change runtime type → Hardware accelerator: T4 GPU`. The first
> code cell below checks that CUDA is actually available and prints the GPU
> info — if it fails, fix the runtime before proceeding.


## 1. Setup

Imports, reproducibility, and device selection. We prefer CUDA when available
(Colab T4), fall back to Apple Silicon's MPS backend for local work, and
finally CPU.


In [ ]:
import os

# Detect Colab so the same notebook also runs locally without modification.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# On Colab we mount Drive and work from a persistent folder so checkpoints
# survive runtime resets; locally we just use the current directory.
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/ELEC5304'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
else:
    PROJECT_DIR = os.getcwd()

print('Working dir:', os.getcwd())

In [ ]:
!nvidia-smi | head -n 20


In [ ]:
import os
import time
import random
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

import matplotlib.pyplot as plt

# Hard-fail early if the runtime isn't on GPU — the time-budgeted Model B
# assumes a T4 and would silently miss its target on CPU.
assert torch.cuda.is_available(), (
    "CUDA is not available. Set Runtime -> Change runtime type -> T4 GPU."
)
device = torch.device('cuda')
print(f'Using device: {device}  ({torch.cuda.get_device_name(0)})')
print(f'torch {torch.__version__} | torchvision {torchvision.__version__}')

In [ ]:
# Fix every RNG we touch so splits, shuffling and init are reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Standard per-channel stats for CIFAR-10 — used by Normalize in the transforms.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
import pickle, subprocess, sys, shutil

# torchvision's CIFAR-10 tarball mirror has been flaky; we instead pull the
# dataset from HuggingFace and rebuild the on-disk layout torchvision expects,
# so the rest of the notebook can use the stock CIFAR10 class unchanged.
DATA_ROOT = '/content/data' if IN_COLAB else os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_ROOT, exist_ok=True)
extracted = os.path.join(DATA_ROOT, 'cifar-10-batches-py')

# Wipe any older cache that wasn't produced by this script (labels key type
# mismatch would crash torchvision's loader).
_marker = os.path.join(extracted, '.hf_strkey_v1')
if os.path.isdir(extracted) and not os.path.isfile(_marker):
    shutil.rmtree(extracted)

if not os.path.isdir(extracted):
    try:
        import datasets
    except ImportError:
        print('Installing `datasets` ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'])
        import datasets

    print('Downloading CIFAR-10 via HuggingFace ...')
    ds = datasets.load_dataset('cifar10')

    def _to_np(split):
        imgs = np.stack([np.array(r['img'], dtype='uint8') for r in split])
        labels = np.array([r['label'] for r in split], dtype='int64')
        return imgs, labels

    x_tr, y_tr = _to_np(ds['train'])
    x_te, y_te = _to_np(ds['test'])

    os.makedirs(extracted, exist_ok=True)

    # Repack into the 5 train + 1 test pickle batches torchvision expects.
    def _dump(name, x, y):
        flat = x.transpose(0, 3, 1, 2).reshape(x.shape[0], -1).astype('uint8')
        with open(os.path.join(extracted, name), 'wb') as f:
            pickle.dump({'data': flat,
                         'labels': [int(v) for v in y.flatten()],
                         'batch_label': name,
                         'filenames': [f'{name}_{i}' for i in range(x.shape[0])]},
                        f)

    for i in range(5):
        _dump(f'data_batch_{i+1}',
              x_tr[i*10000:(i+1)*10000],
              y_tr[i*10000:(i+1)*10000])
    _dump('test_batch', x_te, y_te)

    classes = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
    with open(os.path.join(extracted, 'batches.meta'), 'wb') as f:
        pickle.dump({'label_names': classes,
                     'num_cases_per_batch': 10000,
                     'num_vis': 3072}, f)

    open(_marker, 'w').close()
    print('Reconstructed torchvision on-disk layout OK')

# Disable MD5 integrity check: our reconstructed pickles obviously don't match
# the official hashes, but the contents are identical.
import torchvision.datasets.cifar as _cifar_mod
CIFAR10._check_integrity = lambda self: True
_cifar_mod.check_integrity = lambda *a, **kw: True

_train = CIFAR10(root=DATA_ROOT, train=True,  download=False)
_test  = CIFAR10(root=DATA_ROOT, train=False, download=False)

print(f'Stock train size: {len(_train)}  |  stock test size: {len(_test)}')
print(f'Image shape: {np.array(_train[0][0]).shape}  |  dtype: uint8')

In [ ]:
# Train-time augmentation: pad-then-crop + horizontal flip are the canonical
# CIFAR-10 augmentations. Normalize puts inputs on the same scale BN expects.
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# No randomness at eval — only the deterministic ToTensor + Normalize.
eval_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])


class SplitDataset(torch.utils.data.Dataset):
    """Indexes transparently across the concatenation of train+test pools.

    The assignment asks for an 80/20 split over *all* 60k images, so we merge
    the two official splits and index by a single global id; indices < 50k
    come from the train pool, the rest from the test pool.
    """

    def __init__(self, raw_train, raw_test, indices, transform):
        self.raw_train = raw_train
        self.raw_test  = raw_test
        self.indices   = list(indices)
        self.transform = transform
        self._n_train  = len(raw_train)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        if idx < self._n_train:
            img, y = self.raw_train[idx]
        else:
            img, y = self.raw_test[idx - self._n_train]
        return self.transform(img), y


# Single deterministic shuffle of all 60k ids, then carve out test / val / train.
# We carve val from the 80% training pool so the 20% test set stays untouched
# until the very end of each run.
full_n = len(_train) + len(_test)
gen = torch.Generator().manual_seed(SEED)
all_idx = torch.randperm(full_n, generator=gen).tolist()

n_test  = full_n - int(round(0.8 * full_n))
n_train_pool = full_n - n_test
n_val   = int(round(0.1 * n_train_pool))
n_train = n_train_pool - n_val
train_idx = all_idx[:n_train]
val_idx   = all_idx[n_train:n_train + n_val]
test_idx  = all_idx[n_train + n_val:]

train_set = SplitDataset(_train, _test, train_idx, train_tf)
val_set   = SplitDataset(_train, _test, val_idx,   eval_tf)
test_set  = SplitDataset(_train, _test, test_idx,  eval_tf)

print(f'Split -> train: {len(train_set)}  |  val: {len(val_set)}  |  test: {len(test_set)}')

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2  # Colab free tier only exposes 2 vCPUs.

# pin_memory + persistent_workers cut per-epoch overhead noticeably on T4.
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, drop_last=False)
val_loader   = DataLoader(val_set,   batch_size=256,        shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True)
test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True)

# Sanity-check one batch and show a few samples so we can eyeball that the
# augmentation + normalisation pipeline isn't silently corrupting images.
xb, yb = next(iter(train_loader))
print(f'Batch shape: {xb.shape}  |  labels: {yb.shape}  |  dtype: {xb.dtype}')

def _denorm(t):
    # Invert Normalize so imshow sees the original [0, 1] pixel range.
    mean = torch.tensor(CIFAR_MEAN).view(3,1,1)
    std  = torch.tensor(CIFAR_STD).view(3,1,1)
    return (t * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, img, lbl in zip(axes.flat, xb[:12], yb[:12]):
    ax.imshow(_denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[lbl], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Training utilities

Shared training loop + evaluator used by both Model A and Model B. Keeping
them in a single place means the two models differ only in *architecture* and
*hyperparameters*, which makes the comparison fair.


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    """Average loss + top-1 accuracy over a loader. Used for val and test."""
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    # sum-reduction so we can divide by the exact number of samples at the end.
    crit = nn.CrossEntropyLoss(reduction='sum')
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss_sum += crit(logits, y).item()
        correct  += (logits.argmax(1) == y).sum().item()
        total    += y.size(0)
    return loss_sum / total, correct / total


def train_model(model, epochs, train_loader, val_loader, device,
                optimizer='sgd',
                lr=0.1, weight_decay=5e-4, momentum=0.9, nesterov=True,
                scheduler='cosine', max_lr=None, pct_start=0.1,
                label_smoothing=0.0,
                log_every=1, time_budget_s=None, use_amp=False):
    """Generic train loop shared by both models.

    Only the hyperparameters differ between Model A and Model B, so keeping
    one loop here guarantees an apples-to-apples comparison.
    `time_budget_s` enforces Model B's 60 s wall-clock cap; `use_amp` flips
    on fp16 mixed precision for the extra speed Model B needs.
    """
    model = model.to(device)
    if optimizer == 'sgd':
        opt = optim.SGD(model.parameters(), lr=lr, momentum=momentum,
                        weight_decay=weight_decay, nesterov=nesterov)
    elif optimizer == 'adamw':
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f'unknown optimizer {optimizer!r}')
    crit = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # LR schedulers step per-batch (not per-epoch) so cosine/OneCycle curves
    # resolve smoothly even for very short runs.
    steps_per_epoch = len(train_loader)
    if scheduler == 'cosine':
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps_per_epoch)
    elif scheduler == 'onecycle':
        sch = optim.lr_scheduler.OneCycleLR(
            opt, max_lr=max_lr or lr,
            steps_per_epoch=steps_per_epoch, epochs=epochs,
            pct_start=pct_start, anneal_strategy='cos')
    else:
        sch = None

    # GradScaler is a no-op when amp=False, so always constructing it is safe.
    amp = use_amp and device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    wall_start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        ep_start = time.time()
        loss_sum = correct = total = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            if amp:
                # fp16 forward/backward; scaler.step handles inf/NaN skipping.
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    logits = model(x)
                    loss = crit(logits, y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits = model(x)
                loss = crit(logits, y)
                loss.backward()
                opt.step()
            if sch is not None:
                sch.step()

            loss_sum += loss.item() * y.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            total    += y.size(0)

            # Cut mid-epoch if we've blown the budget (only relevant for Model B).
            if time_budget_s is not None and (time.time() - wall_start) > time_budget_s:
                break

        ep_time = time.time() - ep_start
        train_loss = loss_sum / max(total, 1)
        train_acc  = correct / max(total, 1)
        val_loss, val_acc = evaluate(model, val_loader, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_time'].append(ep_time)

        if epoch % log_every == 0 or epoch == 1 or epoch == epochs:
            print(f'ep {epoch:3d}/{epochs} | '
                  f'train {train_loss:.3f}/{train_acc*100:5.2f}% | '
                  f'val   {val_loss:.3f}/{val_acc*100:5.2f}% | '
                  f'{ep_time:5.1f}s')

        if time_budget_s is not None and (time.time() - wall_start) > time_budget_s:
            print(f'Hit wall-clock budget of {time_budget_s:.0f}s, stopping after epoch {epoch}.')
            break

    history['wall_time'] = time.time() - wall_start
    return history


def plot_history(h, title=''):
    """Side-by-side loss and accuracy curves for train vs val."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(h['train_loss']) + 1)
    axes[0].plot(epochs, h['train_loss'], label='train')
    axes[0].plot(epochs, h['val_loss'],   label='val')
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[0].set_title(f'{title} — loss')
    axes[1].plot(epochs, [a*100 for a in h['train_acc']], label='train')
    axes[1].plot(epochs, [a*100 for a in h['val_acc']],   label='val')
    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=.3)
    axes[1].set_title(f'{title} — accuracy')
    plt.tight_layout()
    plt.show()

## 4. Model A — Target > 80% test accuracy

### Why a ResNet-style CNN?
For CIFAR-10 the community-standard baseline is a *small* residual network.
Residual connections solve the vanishing-gradient problem of deeper plain
CNNs, which in practice means we can train 20–50 layers stably and push past
80% accuracy without much tuning.

We use a compact **ResNet-20**-style architecture (He et al., 2016):

- Stem: `Conv 3→16`, BN, ReLU.
- Three stages of BasicBlocks with widths `[16, 32, 64]`, each stage downsampling 2× via stride-2 conv at its first block.
- Global average pooling, then a `Linear → 10` classifier.
- About **0.27 M parameters** — small enough to train comfortably on MPS and on Colab.

### Training recipe
- SGD + Nesterov momentum 0.9, weight decay 5e-4.
- Initial LR 0.1 with **cosine annealing** to 0 over the full run.
- Batch size 128, 50 epochs.
- Augmentation: RandomCrop(4)+HorizontalFlip (already baked into the train DataLoader).

These are the exact ingredients of the classic "He et al. 2016 short schedule"
which routinely hits ~91% on the standard split — we expect to clear 80% with
healthy margin on our 80/20 split.


In [ ]:
class BasicBlock(nn.Module):
    """Two 3x3 convs + BN with an additive skip connection (He et al. 2016)."""
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        # bias=False because BN right after absorbs the bias term.
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes,    planes, 3, stride=1,      padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        # Identity shortcut unless we need to downsample or change channels —
        # then use a 1x1 conv to match shapes for the add.
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        # Residual add happens *before* the final ReLU, as in the original paper.
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


class ResNetCIFAR(nn.Module):
    """CIFAR-style ResNet: stem -> 3 stages of BasicBlocks -> GAP -> linear.

    With n=3 this is ResNet-20 (6n+2 = 20 weighted layers).
    """
    def __init__(self, n=3, num_classes=10, base_width=16):
        super().__init__()
        self.in_planes = base_width
        self.stem = nn.Sequential(
            nn.Conv2d(3, base_width, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
        )
        # Widths double and spatial size halves at the start of each new stage.
        self.stage1 = self._make_stage(base_width,     n, stride=1)
        self.stage2 = self._make_stage(base_width * 2, n, stride=2)
        self.stage3 = self._make_stage(base_width * 4, n, stride=2)
        # Global average pool drops spatial dims so the classifier is input-size
        # agnostic and parameter-light.
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(base_width * 4, num_classes)
        self._init_weights()

    def _make_stage(self, planes, num_blocks, stride):
        # Only the first block in each stage can change shape.
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def _init_weights(self):
        # He init for conv+ReLU stacks; standard BN and small linear init.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


model_a = ResNetCIFAR(n=3)
n_params = sum(p.numel() for p in model_a.parameters())
print(f'ResNet-20 parameters: {n_params/1e6:.3f} M')

# Quick shape check so a silly mismatch trips here, not 20 minutes into training.
with torch.no_grad():
    print('logits shape:', model_a(torch.randn(2, 3, 32, 32)).shape)

In [ ]:
# Recipe: SGD+Nesterov with cosine-annealed LR over all 50 epochs.
# This is the classic "He 2016 short schedule" that reliably hits 90%+ on CIFAR-10.
EPOCHS_A = 50
history_a = train_model(
    model_a,
    epochs=EPOCHS_A,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    optimizer='sgd',
    lr=0.1,
    weight_decay=5e-4,
    momentum=0.9,
    nesterov=True,
    scheduler='cosine',
    label_smoothing=0.0,
    log_every=1,
)
plot_history(history_a, title='Model A (ResNet-20)')

# Test set is evaluated exactly once, at the end, so the reported number is
# not contaminated by schedule or early-stopping decisions.
test_loss_a, test_acc_a = evaluate(model_a, test_loader, device)
print(f'Best val acc: {max(history_a["val_acc"])*100:.2f}%  |  '
      f'final val: {history_a["val_acc"][-1]*100:.2f}%  |  '
      f'TEST acc: {test_acc_a*100:.2f}%')

## 5. Model B — Target ≥ 70% test accuracy in ≤ 1 minute on a T4 GPU

### Why a different recipe?
With ≤ 60 s on a T4, per-epoch time is CPU-DataLoader-bound on Colab's 2-vCPU
VM, so the real scarce resource is **gradient updates**. Model A's SGD + batch
128 takes ~19 s/epoch → only 3 epochs fit in the budget, and SGD simply
doesn't converge in 3 epochs.

Model B's recipe therefore changes three things at once, matching the
DAWNBench-style "how to train your ResNet" (Page, 2018) findings:

1. **Optimizer: AdamW.** Adaptive per-parameter step sizes converge in a
   handful of epochs where SGD would need dozens.
2. **Smaller batch (128).** Epoch wall-clock is dominated by the CPU data
   pipeline, not the GPU, so shrinking the batch gives ~4× more optimizer
   steps per epoch at essentially the same per-epoch time.
3. **OneCycle schedule sized to what actually runs (3 epochs).** With
   `pct_start=0.2` and `max_lr=0.02`, the LR warms up for ~20% of training
   and decays for the rest, so by the final step the schedule has completed
   rather than being cut off mid-warmup.
4. **Mixed precision (AMP).** Roughly halves per-step GPU cost on CUDA for
   essentially free.
5. **Label smoothing 0.1.** Small regulariser that helps short runs avoid
   overconfident early predictions.

The architecture is a narrow 3-stage plain CNN (FastCIFARNet, width 48,
~1 M params) — large enough to exploit AdamW's fast early progress, small
enough that each forward+backward stays GPU-cheap.


In [ ]:
class FastCIFARNet(nn.Module):
    """Plain (no-residual) 3-stage CNN tuned for short-run super-convergence.

    Width 48 keeps the param count ~1M — big enough to exploit AdamW's fast
    early progress, small enough that each step stays GPU-cheap on a T4.
    """
    def __init__(self, num_classes=10, width=48):
        super().__init__()
        # Local helper: the Conv-BN-ReLU triplet we reuse everywhere.
        def conv_bn(ci, co, stride=1):
            return nn.Sequential(
                nn.Conv2d(ci, co, 3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(co),
                nn.ReLU(inplace=True),
            )

        w = width
        self.stem = conv_bn(3, w)
        # Each stage: one "same" conv then a stride-2 conv that doubles channels
        # and halves spatial size. Third stage keeps width fixed (already at 4w).
        self.stage1 = nn.Sequential(conv_bn(w,    w),    conv_bn(w,    w*2, stride=2))
        self.stage2 = nn.Sequential(conv_bn(w*2,  w*2),  conv_bn(w*2,  w*4, stride=2))
        self.stage3 = nn.Sequential(conv_bn(w*4,  w*4),  conv_bn(w*4,  w*4, stride=2))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(w*4, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        return self.fc(self.pool(x).flatten(1))


model_b = FastCIFARNet(width=48)
print(f'FastCIFARNet parameters: {sum(p.numel() for p in model_b.parameters())/1e3:.1f} k')

In [ ]:
# Separate loaders: smaller train batch = more optimizer steps per epoch
# (the CPU DataLoader, not the GPU, is the bottleneck on Colab's 2 vCPUs),
# and a big eval batch keeps per-epoch val/test overhead negligible.
FAST_BATCH = 128
fast_train_loader = DataLoader(train_set, batch_size=FAST_BATCH, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               persistent_workers=True, drop_last=True)
fast_val_loader   = DataLoader(val_set,   batch_size=1024, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               persistent_workers=True)
fast_test_loader  = DataLoader(test_set,  batch_size=1024, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               persistent_workers=True)

# Recipe: AdamW + OneCycle sized to 3 epochs + AMP + label smoothing.
# time_budget_s=60 enforces the T4 one-minute cap from the brief.
EPOCHS_B = 3
history_b = train_model(
    model_b,
    epochs=EPOCHS_B,
    train_loader=fast_train_loader,
    val_loader=fast_val_loader,
    device=device,
    optimizer='adamw',
    lr=0.01, weight_decay=1e-4,
    scheduler='onecycle', max_lr=0.02, pct_start=0.2,
    label_smoothing=0.1,
    log_every=1,
    time_budget_s=60,
    use_amp=True,
)
plot_history(history_b, title='Model B (FastCIFARNet)')

test_loss_b, test_acc_b = evaluate(model_b, fast_test_loader, device)
print(f'Best val acc: {max(history_b["val_acc"])*100:.2f}%  |  '
      f'wall-clock: {history_b["wall_time"]:.1f}s  |  '
      f'TEST acc: {test_acc_b*100:.2f}%')

## 6. Per-class accuracy & confusion matrices

Top-1 accuracy alone hides which classes the network struggles with. Below we
report the confusion matrix and per-class accuracy on the **held-out test
set** (12 000 images) for both models. CIFAR-10's classic confusion pair is
*cat* vs *dog*; we expect the matrices to reflect that.

In [ ]:
@torch.no_grad()
def collect_preds(model, loader, device):
    """Run the model over a loader and return (true_labels, predicted_labels)."""
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        ps.append(model(x).argmax(1).cpu())
        ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


def confusion_matrix(y_true, y_pred, num_classes=10):
    # Rows = true class, cols = predicted class. Diagonal = correct predictions.
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def report(name, model, loader):
    """Print overall + per-class accuracy and return the confusion matrix."""
    y_true, y_pred = collect_preds(model, loader, device)
    cm = confusion_matrix(y_true, y_pred)
    # Per-class recall: diag / row sum.
    per_class = cm.diagonal() / cm.sum(axis=1)
    overall = (y_true == y_pred).mean()

    print(f'\n=== {name}  |  overall test acc: {overall*100:.2f}% ===')
    print(f'{"class":<12}{"acc":>8}   support')
    for i, c in enumerate(CLASS_NAMES):
        print(f'{c:<12}{per_class[i]*100:7.2f}%   {cm[i].sum():>5d}')
    worst = int(np.argmin(per_class))
    best  = int(np.argmax(per_class))
    print(f'best:  {CLASS_NAMES[best]} ({per_class[best]*100:.2f}%)  |  '
          f'worst: {CLASS_NAMES[worst]} ({per_class[worst]*100:.2f}%)')
    return cm, per_class


def plot_cm(cm, title):
    # Row-normalise so each row sums to 1 — classes have slightly different
    # sample counts under the random split and raw counts would mislead.
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(title)
    # Annotate each cell; flip text colour on dark cells for readability.
    for i in range(10):
        for j in range(10):
            v = cm_norm[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.5 else 'black', fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


cm_a, pc_a = report('Model A (ResNet-20)',     model_a, test_loader)
plot_cm(cm_a, 'Model A — confusion matrix (row-normalised)')

cm_b, pc_b = report('Model B (FastCIFARNet)',  model_b, fast_test_loader)
plot_cm(cm_b, 'Model B — confusion matrix (row-normalised)')

## 6. Task 2 — Question 1

**Q. What kind of neural network architecture is more suitable for learning
under limited training time? Why?**

When training time is the scarce resource (rather than data or parameters),
the architectures that win are those whose **inductive bias matches the task
so closely that very few gradient updates are needed to reach "good enough"
accuracy**. For image classification specifically, this points to a compact
**convolutional** network with:

1. **Strong spatial priors.** Convolutions share weights across positions and
   only connect local neighbourhoods, so every image in a batch contributes
   gradient signal to *every* filter. A fully-connected network or a
   transformer has to *learn* this prior from data — burning precious epochs.
2. **BatchNorm after every conv.** BN stabilises the loss landscape and lets
   us crank the learning rate up by an order of magnitude, turning raw
   wall-clock seconds into more effective gradient updates per unit time.
3. **Residual or short skip connections.** They keep gradients well-scaled in
   deeper networks, so we can add depth (which is cheap per-step on a GPU)
   without training grinding to a halt.
4. **A small parameter count matched to the compute budget.** Doubling params
   roughly doubles step time. With only ~60 s on a T4 the sweet spot is a
   narrow CNN in the 100–300 k parameter range — enough capacity for CIFAR-10
   but small enough to fit ten or so epochs at batch 512.
5. **A super-convergent LR schedule (OneCycle).** Pairing a small CNN with a
   high-LR, short-cycle schedule (Smith 2018; Page 2018 "How to train your
   ResNet") routinely reaches 70%+ on CIFAR-10 in tens of seconds on a T4.
6. **Mixed-precision (fp16) arithmetic.** On CUDA this alone cuts per-step
   time by ~1.5–2× for the same quality of gradient.

In short: a **small BN-regularised residual/plain CNN trained with a
super-convergence schedule under mixed precision** is the right tool for a
tight time budget. Bigger models (deep ResNet-50, Vision Transformers) can
eventually reach higher accuracy, but every extra parameter is paid for in
seconds, and they usually also need much more data-augmentation warm-up to
beat a small CNN's head-start.


## 7. Results summary

**Split:** train 43 200 · val 4 800 · test 12 000. Val is used for per-epoch
monitoring; the test set is evaluated **once**, at the very end of each run.

| Model | Params | Schedule | Epochs | Best val acc. | Final test acc. | Notes |
|---|---|---|---|---|---|---|
| **A — ResNet-20** | ~0.27 M | SGD + cosine, 50 ep, BS 128 | 50 | **91.79%** | **91.31%** | Target > 80% ✅ (+11.31 pts) |
| **B — FastCIFARNet** | ~0.98 M (width 48) | AdamW + OneCycle, AMP, BS 128 | 3 | **77.67%** | **77.09%** | Target ≥ 70% in ≤ 60 s on T4 ✅ (wall-clock 57.3 s, +7.09 pts) |

Both objectives met with healthy margin. Classic CIFAR-10 confusion on *cat*
(worst class in both models: 80.47% for A, 56.35% for B); best-predicted
classes are *ship*/*automobile*, which have the most distinctive shape/colour
cues in the dataset.
